Calculate heats of formation from a basis.

In [ ]:
import pint
import re
import numpy as np

term_pattern = re.compile(r"([A-Z][a-z]?)(\d*)")

elements = ("C", "H", "O")
basis = ("H2", "CH4", "H2O")

# All units in Hartree (converted from kcal/mol)
basis_formation_enthalpy = {
    "H2": 0.0 * 0.0015936,
    "CH4": -15.906 * 0.0015936,
    "H2O": -57.106 * 0.0015936,
}


def stoichiometry(formula: str) -> dict[str, int]:
    """Parse stoichiometry from formula."""
    matches = term_pattern.findall(formula)
    stoichiometry = {}
    for element, count in matches:
        count = int(count) if count else 1
        stoichiometry[element] = count
    return stoichiometry


def stoichiometry_vector(formula: str) -> list[int]:
    """Get stoichiometry vector for a given formula."""
    stoich = stoichiometry(formula)
    return [stoich.get(element, 0) for element in elements]

basis_matrix = np.column_stack(tuple(stoichiometry_vector(formula) for formula in basis))


def basis_coefficients(formula: str) -> dicst[str, int]:
    """Calculate basis coefficients for a given formula."""
    coeffs =  np.linalg.solve(basis_matrix, stoichiometry_vector(formula))
    return {k: c for k, c in zip(basis, coeffs.tolist(), strict=True)}


def formation_enthalpy(formula: str, absolute_enthalpy: float, basis_absolute_enthalpy: dict[str, float]) -> float:
    """Calculate the shift from electronic energy to heat of formation."""
    coeff = basis_coefficients(formula)
    reference_correction = sum(coeff[k] * (basis_formation_enthalpy[k] - basis_absolute_enthalpy[k]) for k in basis)
    return absolute_enthalpy + reference_correction

H_rh = 0.020172108000025446
H_oh = 0.012206926400011753
H_r = 0.0874287760000243


In [20]:
# My results:
basis_absolute_enthalpy = {
    "H2": -1.163338874,
    "CH4": -40.40372913,
    "H2O": -76.33508129,
}

H_rh = formation_enthalpy("C5H8", -194.8920249, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_rh = }")
H_oh = formation_enthalpy("OH", -75.64980371, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_oh = }")
H_c1_c2 = formation_enthalpy("C5H9O", -270.5457024, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c1_c2 = }")
H_c2_rhoh = formation_enthalpy("C5H9O", -270.5471261, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c2_rhoh = }")
H_c1_alkyl = formation_enthalpy("C5H9O", -270.5414663, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c1_alkyl = }")
H_c1_allyl = formation_enthalpy("C5H9O", -270.5432814, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c1_allyl = }")
H_c1_vinyl = formation_enthalpy("C5H9O", -270.5342003, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c1_vinyl = }")
H_rhoh = formation_enthalpy("C5H9O", -270.5870779, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_rhoh = }")
H_c1 = formation_enthalpy("C5H9O", -270.5462542, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c1 = }")
H_c2 = formation_enthalpy("C5H9O", -270.5464714, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_c2 = }")
# H_r = formation_enthalpy("C5H7", -194.2411929, basis_absolute_enthalpy=basis_absolute_enthalpy)
# print(f"{H_r = }")

H_rh = 0.01984849800001598
H_oh = 0.012604021400008492
H_c1_c2 = 0.02857872940001016
H_c2_rhoh = 0.027155029399978048
H_c1_alkyl = 0.03281482939996749
H_c1_allyl = 0.03099972939997997
H_c1_vinyl = 0.04008082939998303
H_rhoh = -0.012796770600004947
H_c1 = 0.028026929399970868
H_c2 = 0.027809729400019023


In [ ]:
# MechDriver results:
basis_absolute_enthalpy = {
    "H2": -1.163339464,
    "CH4": -40.40341344,
    "H2O": -76.33468352,
}

H_rh = formation_enthalpy("C5H8", -194.8901193, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_rh = }")
H_oh = formation_enthalpy("OH", -75.64980274, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_oh = }")
H_r = formation_enthalpy("C5H7", -194.2411929, basis_absolute_enthalpy=basis_absolute_enthalpy)
print(f"{H_r = }")